# Detecting Missing Timestamps

In [ ]:
import numpy as np
import pandas as pd

In [52]:
df = pd.read_csv("sales_with_gaps.csv")
df["date"] = pd.to_datetime(df["date"])

In [53]:
df.set_index("date", inplace=True)

In [54]:
df.head()

,sales
date,
2022-01-01,19.30
2022-01-02,18.69
2022-01-04,20.41
2022-01-05,18.68
2022-01-07,26.24


In [55]:
len(df)

349

### Check 1 - Does the index have a regular frequency

In [56]:
print(f"Frequency: {df.index.freq}")
# None means pandas could not infer a regular frequency - red flag

Frequency: None


### Check 2 - Build the complete expected date range

In [57]:
expected = pd.date_range(start = df.index.min(), end=df.index.max(), freq="D")

In [58]:
print(f"Expected rows: {len(expected)}")
print(f"Actual rows: {len(df)}")
print(f"Missing rows: {len(expected) - len(df)}")

Expected rows: 365
Actual rows: 349
Missing rows: 16


### Check 3 - Find exactly which timestamps are missing

In [59]:
missing_dates = expected.difference(df.index)
print(f"Missing Timestamps ({len(missing_dates)} total):")
print(missing_dates.tolist())

Missing Timestamps (16 total):
[Timestamp('2022-01-03 00:00:00'), Timestamp('2022-01-06 00:00:00'), Timestamp('2022-02-14 00:00:00'), Timestamp('2022-03-20 00:00:00'), Timestamp('2022-03-21 00:00:00'), Timestamp('2022-03-22 00:00:00'), Timestamp('2022-05-08 00:00:00'), Timestamp('2022-07-04 00:00:00'), Timestamp('2022-09-10 00:00:00'), Timestamp('2022-09-11 00:00:00'), Timestamp('2022-09-12 00:00:00'), Timestamp('2022-09-13 00:00:00'), Timestamp('2022-11-25 00:00:00'), Timestamp('2022-11-26 00:00:00'), Timestamp('2022-12-25 00:00:00'), Timestamp('2022-12-26 00:00:00')]


# Reindexing to a Complete Timeline

Once you know timestamps are missing, you reindex the dataframe to the full expected range. This converts missing timestamps into rows with NaN values — making the gap explicit and visible rather than hidden.

In [60]:
df_reindexed = df.reindex(expected)
print(df_reindexed.head())

# This is the critical step. Before reindexing, pandas had no idea anything was wrong. Now the gaps are explicit and you can handle them deliberately.

            sales
2022-01-01  19.30
2022-01-02  18.69
2022-01-03    NaN
2022-01-04  20.41
2022-01-05  18.68


# Filling the Gaps

Now you have NaN values where timestamps were missing. The filling strategy depends on what the missing data represents in your domain.

#### Forward Fill - ffill() --> Carries the last known value forward. Best when the value is expected to persist until something changes — sensor readings, prices, inventory levels.

#### Backward Fill - bfill() --> Fills with the next known value. Rarely used alone — mainly useful when you know the next observation is more representative than the previous one.

#### Linear interpolation --> Draws a straight line between the two surrounding known values. Best for smoothly varying series where you expect gradual change between observations.

In [61]:
# ffill()
df_ffill = df_reindexed["sales"].ffill()

In [62]:
# bfill()
df_bfill = df_reindexed["sales"].bfill()

In [63]:
# Interpolation
df_interp = df_reindexed["sales"].interpolate(method="linear")

# was_missing

In [64]:
df_reindexed["was_missing"] = df_reindexed["sales"].isna().astype(int)

# Now fill
df_reindexed["sales"] = df_reindexed["sales"].interpolate(method="linear")

In [65]:
df_reindexed.head()

,sales,was_missing
2022-01-01,19.30,0
2022-01-02,18.69,0
2022-01-03,19.55,1
2022-01-04,20.41,0
2022-01-05,18.68,0
